### Evaluation
This file creates tex documents with several model evaluation tables. Not strategy performance metrics.




In [1]:
import sys, os
sys.path.insert(0, "/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")
sys.modules.pop("src", None)
os.chdir("/Users/shah/CODE_BOOK_4/THESIS_WORKING/THESIS_2")

In [2]:
import pandas as pd
import numpy as np
from math import sqrt, pi, exp
from arch import arch_model
import yfinance as yf 
from scipy.optimize import minimize
import warnings
from arch.__future__ import reindexing
from arch.utility.exceptions import ConvergenceWarning
from src.msGarch import msGARCH
from src.metrics import metrics
from src.msGarch import msGARCHIV
from tqdm import tqdm

from xgboost import XGBRegressor
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message="y is poorly scaled")
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)


In [3]:
# import sys
# !{sys.executable} -m pip install xgboost

## 2019 - 2022

### Load outputs by year and X_q parameters by quarter.

In [4]:
import pickle
outputs_by_year = pickle.load(open("../outputs_by_year/2019_2022/outputs_by_year_12mdls.pkl", "rb"))

In [5]:
# outputs_by_year[2019]['q1'][['rv', 'ms_base', 'ms_LSTM_noIV', 'ms_LSTM_wIV']][1300:1500].plot(figsize=(16,5), title = "RV, ms-base, ms-gatedIV")


In [6]:

import pandas as pd
from pathlib import Path

INDIR = Path("../data/readyData")
YEARS = [2019, 2020, 2021]

X_by_year = {y: pd.read_parquet(INDIR / f"d_X_{y}.parquet") for y in YEARS}


In [7]:
import numpy as np

eps = 1e-12
for y, X in X_by_year.items():
    X_by_year[y][["bsIV","MFIV"]] = np.exp(
        np.log(X[["bsIV","MFIV"]].clip(lower=eps))
          .interpolate(method="time", limit_direction="both")
    )


In [8]:
# separated nested dict with year(2002) and quarters("q1") 
# X  : rt , bsIV , MFIV
Xq_by_year = {}

for y, X in X_by_year.items():
    Xq_by_year[y] = {
        "q1": X.loc[f"{y}-01-01":f"{y}-03-31 23:59:59"],
        "q2": X.loc[f"{y}-04-01":f"{y}-06-30 23:59:59"],
        "q3": X.loc[f"{y}-07-01":f"{y}-09-30 23:59:59"],
        "q4": X.loc[f"{y}-10-01":f"{y}-12-31 23:59:59"],
    }

# example: Xq_by_year[2025]["q1"]


In [9]:
# these are forecasts made by each model at t-1 aligned with t  
Xq_by_year[2019]["q1"].columns

Index(['rt', 'bsIV', 'MFIV'], dtype='str')

### Load Saved MSGarch Outputs

In [10]:

#LOAD BASE 
import pickle
globals().update(pickle.load(open("../model_outputs_19_22/ms_garch_base_outputs.pkl","rb")))


In [12]:

#LOAD BASE 
import pickle
globals().update(pickle.load(open("../model_outputs_19_22/ms_garch_gatedIV_outputs.pkl","rb")))


In [14]:

#LOAD BASE 
import pickle
globals().update(pickle.load(open("../model_outputs_19_22/ms_garch_noIV_outputs.pkl","rb")))

In [16]:
[k for k in globals() if k.startswith("h_final_noIV_by_year")]

['h_final_noIV_by_year']

### Evaluation

In [17]:
def metrics_new(y_true, y_pred, eps=1e-10):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)

    m = min(len(y_true), len(y_pred))
    y_true = np.clip(y_true[-m:], eps, None)
    y_pred = np.clip(y_pred[-m:], eps, None)

    # Errors
    err = y_true - y_pred

    # Core
    mse  = np.mean(err**2)
    rmse = np.sqrt(mse)
    mae  = np.mean(np.abs(err))
    bias = np.mean(err)

    # Likelihood / variance metrics
    qlike = np.mean(np.log(y_pred) + y_true / y_pred)
    nll   = 0.5 * np.mean(np.log(2*np.pi) + np.log(y_pred) + y_true / y_pred)

    # Relative metrics
    hmse = np.mean((1 - y_true / y_pred)**2)
    hmae = np.mean(np.abs(1 - y_true / y_pred))

    # R2
    r2 = 1 - np.sum(err**2) / np.sum((y_true - np.mean(y_true))**2)

    # MADL (variance → signal)
    signal = y_pred - np.mean(y_pred)
    direction = np.sign(signal)
    madl = np.mean(-np.sign(y_true * direction) * np.abs(y_true))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MSE": mse,
        "NLL": nll,
        "QLIKE": qlike,
        "HMSE": hmse,
        "HMAE": hmae,
        "R2": r2,
        "Bias": bias,
        "MADL": madl
    }

In [18]:

quarters = [(y, q, Xq_by_year[y][q]) for y in sorted(Xq_by_year.keys()) for q in ["q1","q2","q3","q4"]]

tables_all = {}

for y, q, _ in quarters:
    results = {}

    rv2_q = np.asarray(rv2_gatedIV_by_year[y][q], float)

    # NEW: directly use dataframe columns
    df_q = outputs_by_year[y][q]

    ms_base_q      = df_q["ms_base"].to_numpy(float)
    ms_LSTM_noIV_q = df_q["ms_LSTM_noIV"].to_numpy(float)
    ms_LSTM_wIV_q  = df_q["ms_LSTM_wIV"].to_numpy(float)

    target_idx = df_q.index[-len(rv2_q):]

    garch_q    = df_q["garch"].reindex(target_idx).to_numpy(float)
    xgboost_q  = df_q["xg_boost"].reindex(target_idx).to_numpy(float)
    sv_q       = df_q["sv"].reindex(target_idx).to_numpy(float)
    gjr_q      = df_q["gjr"].reindex(target_idx).to_numpy(float)
    egarch_q   = df_q["egarch"].reindex(target_idx).to_numpy(float)
    tarch_q    = df_q["tarch"].reindex(target_idx).to_numpy(float)
    midas_q    = df_q["midas"].reindex(target_idx).to_numpy(float)
    garch_m_q  = df_q["garch_m"].reindex(target_idx).to_numpy(float)

    valid = (
        np.isfinite(rv2_q)
        & np.isfinite(ms_base_q)
        & np.isfinite(ms_LSTM_noIV_q)
        & np.isfinite(ms_LSTM_wIV_q)
        & np.isfinite(garch_q)
        & np.isfinite(xgboost_q)
        & np.isfinite(sv_q)
        & np.isfinite(gjr_q)
        & np.isfinite(egarch_q)
        & np.isfinite(tarch_q)
        & np.isfinite(midas_q)
        & np.isfinite(garch_m_q)
    )

    rv2_use         = rv2_q[valid]
    ms_base_use     = ms_base_q[valid]
    ms_LSTM_noIV_use= ms_LSTM_noIV_q[valid]
    ms_LSTM_wIV_use = ms_LSTM_wIV_q[valid]

    garch_use    = garch_q[valid]
    xgboost_use  = xgboost_q[valid]
    sv_use       = sv_q[valid]
    gjr_use      = gjr_q[valid]
    egarch_use   = egarch_q[valid]
    tarch_use    = tarch_q[valid]
    midas_use    = midas_q[valid]
    garch_m_use  = garch_m_q[valid]

    results["GARCH_M"]      = metrics_new(rv2_use, garch_m_use)
    results["MS_base"]      = metrics_new(rv2_use, ms_base_use)
    results["MS_LSTM_noIV"] = metrics_new(rv2_use, ms_LSTM_noIV_use)
    results["MS_LSTM_wIV"]  = metrics_new(rv2_use, ms_LSTM_wIV_use)
    results["GARCH"]        = metrics_new(rv2_use, garch_use)
    results["XGBoost"]      = metrics_new(rv2_use, xgboost_use)
    results["SV"]           = metrics_new(rv2_use, sv_use)
    results["GJR"]          = metrics_new(rv2_use, gjr_use)
    results["EGARCH"]       = metrics_new(rv2_use, egarch_use)
    results["TARCH"]        = metrics_new(rv2_use, tarch_use)
    results["MIDAS"]        = metrics_new(rv2_use, midas_use)

    metric_order = ["RMSE","MAE","MSE","NLL","QLIKE","HMSE","HMAE","R2","Bias","MADL"]
    table = pd.DataFrame(results).T[metric_order].round(6)

    if y not in tables_all:
        tables_all[y] = {}

    tables_all[y][q] = table

    with open(f"tables/IS/model_evaluation/metrics_{str(y)[2:]}_{q}.tex", "w") as f:
        f.write(table.to_latex(index=True, float_format="%.6f"))

In [19]:
all_q = pd.concat(
    [tables_all[y][q].assign(Year=y, Quarter=q) for y in tables_all for q in tables_all[y]],
    keys=[f"{y}_{q}" for y in tables_all for q in tables_all[y]]
)

cum_metrics = all_q.drop(columns=["Year", "Quarter"]).groupby(level=1).mean().round(6)
cum_std     = all_q.drop(columns=["Year", "Quarter"]).groupby(level=1).std().round(6)

### Cumulative Metrics across Quarters + Stdev of errors


In [20]:
print("Cumulative Metrics Across Models and quarters")
cum_metrics


Cumulative Metrics Across Models and quarters


,RMSE,MAE,MSE,NLL,QLIKE,HMSE,HMAE,R2,Bias,MADL
EGARCH,0.000416,0.000092,0.000000,-3.196654,-8.231185,847.082931,2.379029,-5.130200e-02,0.000000,-0.000000
GARCH,0.046267,0.001081,0.013454,-3.043571,-7.925019,1534.556614,2.748794,-1.030867e+04,-0.000986,0.000028
GARCH_M,0.000537,0.000106,0.000000,-3.505146,-8.848170,101.699107,1.442811,-1.936120e+00,-0.000020,0.000008
GJR,0.000416,0.000092,0.000000,-3.196654,-8.231185,847.082931,2.379029,-5.130200e-02,0.000000,-0.000000
MIDAS,0.000420,0.000093,0.000000,-3.145962,-8.129802,1258.802199,2.486961,-6.727800e-02,0.000000,0.000001
MS_LSTM_noIV,0.000425,0.000088,0.000000,-2.817986,-7.473849,2888.831110,3.294882,-9.883100e-02,0.000007,0.000002
MS_LSTM_wIV,0.000410,0.000086,0.000000,-3.405699,-8.649275,197.924682,1.816419,-2.872900e-02,0.000008,-0.000003
MS_base,0.000455,0.000104,0.000000,-3.124234,-8.086346,1518.539365,2.475473,-2.334600e-01,-0.000016,0.000002
SV,14.102216,0.634823,781.658743,-2.209704,-6.257286,12135.545698,3.191830,-2.132282e+10,-0.634766,0.000061
TARCH,0.000416,0.000092,0.000000,-3.196654,-8.231185,847.082931,2.379029,-5.130200e-02,0.000000,-0.000000


In [ ]:
print("Standard deviation of metrics across Models and quarters")
cum_std

Cumulative Metrics Across Models and quarters


,RMSE,MAE,MSE,NLL,QLIKE,HMSE,HMAE,R2,Bias,MADL
EGARCH,0.000435,0.000058,0.000001,0.389580,0.779161,2152.688629,0.882576,3.546600e-02,0.000000,0.000011
GARCH,0.111096,0.002435,0.035092,0.500441,1.000882,4229.008307,1.060255,2.542464e+04,0.002386,0.000047
GARCH_M,0.000437,0.000063,0.000001,0.305837,0.611674,147.375313,0.252390,3.208814e+00,0.000031,0.000023
GJR,0.000435,0.000058,0.000001,0.389580,0.779161,2152.688629,0.882576,3.546600e-02,0.000000,0.000011
MIDAS,0.000443,0.000059,0.000001,0.445014,0.890027,3518.278082,0.995996,4.034300e-02,0.000000,0.000010
MS_LSTM_noIV,0.000447,0.000059,0.000001,0.702009,1.404017,7970.073951,1.667066,1.069590e-01,0.000011,0.000013
MS_LSTM_wIV,0.000429,0.000053,0.000001,0.281944,0.563887,298.765175,0.610396,6.061600e-02,0.000021,0.000015
MS_base,0.000498,0.000073,0.000001,0.499957,0.999914,4294.029260,1.173189,2.172180e-01,0.000025,0.000014
SV,25.214417,1.152470,1803.257360,1.170969,2.341939,37922.726771,2.380546,5.776874e+10,1.152458,0.000039
TARCH,0.000435,0.000058,0.000001,0.389580,0.779161,2152.688629,0.882576,3.546600e-02,0.000000,0.000011


In [22]:
with open("tables/IS/cumulative_model_evaluation/cumulative_metrics.tex", "w") as f:
    f.write(cum_metrics.to_latex(index=True, float_format="%.6f"))
with open("tables/IS/cumulative_model_evaluation/std_dev_of_metrics.tex", "w") as f:
    f.write(cum_std.to_latex(index=True, float_format="%.6f"))


## 2022 - 2025

### Load outputs by year and X_q parameters by quarter.

In [23]:
import pickle
outputs_by_year = pickle.load(open("../outputs_by_year/2022_2025/outputs_by_year_12mdls.pkl", "rb"))


In [24]:

INDIR = Path("../data/readyData")
YEARS = [2022, 2023, 2024, 2025]


In [25]:
X_by_year = {y: pd.read_parquet(INDIR / f"d_X_{y}.parquet") for y in YEARS}

eps = 1e-12
for y, X in X_by_year.items():
    X_by_year[y][["bsIV","MFIV"]] = np.exp(
        np.log(X[["bsIV","MFIV"]].clip(lower=eps))
          .interpolate(method="time", limit_direction="both")
    )

# separated nested dict with year(2002) and quarters("q1") 
# X  : rt , bsIV , MFIV
Xq_by_year = {}

for y, X in X_by_year.items():
    Xq_by_year[y] = {
        "q1": X.loc[f"{y}-01-01":f"{y}-03-31 23:59:59"],
        "q2": X.loc[f"{y}-04-01":f"{y}-06-30 23:59:59"],
        "q3": X.loc[f"{y}-07-01":f"{y}-09-30 23:59:59"],
        "q4": X.loc[f"{y}-10-01":f"{y}-12-31 23:59:59"],
    }


### Load globals

In [26]:

#LOAD BASE 
import pickle
globals().update(pickle.load(open("../model_outputs_22_25/ms_garch_base_outputs.pkl","rb")))


#LOAD BASE 
import pickle
globals().update(pickle.load(open("../model_outputs_22_25/ms_garch_gatedIV_outputs.pkl","rb")))

#LOAD BASE 
import pickle
globals().update(pickle.load(open("../model_outputs_22_25/ms_garch_noIV_outputs.pkl","rb")))


### Evaluation

In [27]:
def metrics_new(y_true, y_pred, eps=1e-10):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)

    m = min(len(y_true), len(y_pred))
    y_true = np.clip(y_true[-m:], eps, None)
    y_pred = np.clip(y_pred[-m:], eps, None)

    # Errors
    err = y_true - y_pred

    # Core
    mse  = np.mean(err**2)
    rmse = np.sqrt(mse)
    mae  = np.mean(np.abs(err))
    bias = np.mean(err)

    # Likelihood / variance metrics
    qlike = np.mean(np.log(y_pred) + y_true / y_pred)
    nll   = 0.5 * np.mean(np.log(2*np.pi) + np.log(y_pred) + y_true / y_pred)

    # Relative metrics
    hmse = np.mean((1 - y_true / y_pred)**2)
    hmae = np.mean(np.abs(1 - y_true / y_pred))

    # R2
    r2 = 1 - np.sum(err**2) / np.sum((y_true - np.mean(y_true))**2)

    # MADL (variance → signal)
    signal = y_pred - np.mean(y_pred)
    direction = np.sign(signal)
    madl = np.mean(-np.sign(y_true * direction) * np.abs(y_true))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MSE": mse,
        "NLL": nll,
        "QLIKE": qlike,
        "HMSE": hmse,
        "HMAE": hmae,
        "R2": r2,
        "Bias": bias,
        "MADL": madl
    }


In [28]:

quarters = [(y, q, Xq_by_year[y][q]) for y in sorted(Xq_by_year.keys()) for q in ["q1","q2","q3","q4"]]

tables_all = {}

for y, q, _ in quarters:
    results = {}

    rv2_q = np.asarray(rv2_gatedIV_by_year[y][q], float)

    # NEW: directly use dataframe columns
    df_q = outputs_by_year[y][q]

    ms_base_q      = df_q["ms_base"].to_numpy(float)
    ms_LSTM_noIV_q = df_q["ms_LSTM_noIV"].to_numpy(float)
    ms_LSTM_wIV_q  = df_q["ms_LSTM_wIV"].to_numpy(float)

    target_idx = df_q.index[-len(rv2_q):]

    garch_q    = df_q["garch"].reindex(target_idx).to_numpy(float)
    xgboost_q  = df_q["xg_boost"].reindex(target_idx).to_numpy(float)
    sv_q       = df_q["sv"].reindex(target_idx).to_numpy(float)
    gjr_q      = df_q["gjr"].reindex(target_idx).to_numpy(float)
    egarch_q   = df_q["egarch"].reindex(target_idx).to_numpy(float)
    tarch_q    = df_q["tarch"].reindex(target_idx).to_numpy(float)
    midas_q    = df_q["midas"].reindex(target_idx).to_numpy(float)
    garch_m_q  = df_q["garch_m"].reindex(target_idx).to_numpy(float)

    valid = (
        np.isfinite(rv2_q)
        & np.isfinite(ms_base_q)
        & np.isfinite(ms_LSTM_noIV_q)
        & np.isfinite(ms_LSTM_wIV_q)
        & np.isfinite(garch_q)
        & np.isfinite(xgboost_q)
        & np.isfinite(sv_q)
        & np.isfinite(gjr_q)
        & np.isfinite(egarch_q)
        & np.isfinite(tarch_q)
        & np.isfinite(midas_q)
        & np.isfinite(garch_m_q)
    )

    rv2_use         = rv2_q[valid]
    ms_base_use     = ms_base_q[valid]
    ms_LSTM_noIV_use= ms_LSTM_noIV_q[valid]
    ms_LSTM_wIV_use = ms_LSTM_wIV_q[valid]

    garch_use    = garch_q[valid]
    xgboost_use  = xgboost_q[valid]
    sv_use       = sv_q[valid]
    gjr_use      = gjr_q[valid]
    egarch_use   = egarch_q[valid]
    tarch_use    = tarch_q[valid]
    midas_use    = midas_q[valid]
    garch_m_use  = garch_m_q[valid]

    results["GARCH_M"]      = metrics_new(rv2_use, garch_m_use)
    results["MS_base"]      = metrics_new(rv2_use, ms_base_use)
    results["MS_LSTM_noIV"] = metrics_new(rv2_use, ms_LSTM_noIV_use)
    results["MS_LSTM_wIV"]  = metrics_new(rv2_use, ms_LSTM_wIV_use)
    results["GARCH"]        = metrics_new(rv2_use, garch_use)
    results["XGBoost"]      = metrics_new(rv2_use, xgboost_use)
    results["SV"]           = metrics_new(rv2_use, sv_use)
    results["GJR"]          = metrics_new(rv2_use, gjr_use)
    results["EGARCH"]       = metrics_new(rv2_use, egarch_use)
    results["TARCH"]        = metrics_new(rv2_use, tarch_use)
    results["MIDAS"]        = metrics_new(rv2_use, midas_use)

    metric_order = ["RMSE","MAE","MSE","NLL","QLIKE","HMSE","HMAE","R2","Bias","MADL"]
    table = pd.DataFrame(results).T[metric_order].round(6)

    if y not in tables_all:
        tables_all[y] = {}

    tables_all[y][q] = table

    with open(f"tables/OOS/model_evaluation/metrics_{str(y)[2:]}_{q}.tex", "w") as f:
        f.write(table.to_latex(index=True, float_format="%.6f"))

In [29]:
all_q = pd.concat(
    [tables_all[y][q].assign(Year=y, Quarter=q) for y in tables_all for q in tables_all[y]],
    keys=[f"{y}_{q}" for y in tables_all for q in tables_all[y]]
)

cum_metrics = all_q.drop(columns=["Year", "Quarter"]).groupby(level=1).mean().round(6)
cum_std     = all_q.drop(columns=["Year", "Quarter"]).groupby(level=1).std().round(6)

In [30]:
with open("tables/OOS/cumulative_model_evaluation/cumulative_metrics.tex", "w") as f:
    f.write(cum_metrics.to_latex(index=True, float_format="%.6f"))
with open("tables/OOS/cumulative_model_evaluation/std_dev_of_metrics.tex", "w") as f:
    f.write(cum_std.to_latex(index=True, float_format="%.6f"))
